In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -U mlflow pyngrok xgboost

In [ ]:
import mlflow
print("MLflow version:", mlflow.__version__)

In [ ]:
import os
from pathlib import Path

# Use absolute paths so both the Colab client and MLflow server
# always use the same SQLite database and artifact directory.
BASE_DIR = Path("/content")
DB_PATH = BASE_DIR / "mlflow.db"
ARTIFACT_DIR = BASE_DIR / "mlartifacts"

DB_URI = f"sqlite:///{DB_PATH}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", os.getcwd())
print("MLflow database:", DB_PATH)
print("Artifact directory:", ARTIFACT_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
import time
import os
import json
import warnings
from pathlib import Path
from getpass import getpass

import mlflow
import mlflow.sklearn
from pyngrok import ngrok

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

print("Imports completed.")

In [ ]:
# MLflow configuration
MLFLOW_HOST = "0.0.0.0"
MLFLOW_PORT = 5001
MLFLOW_LOCAL_URL = f"http://127.0.0.1:{MLFLOW_PORT}"
EXPERIMENT_NAME = "employee_attrition"

print("Experiment:", EXPERIMENT_NAME)
print("Local MLflow URL:", MLFLOW_LOCAL_URL)

In [ ]:
# Stop old MLflow/ngrok processes from previous Colab attempts.
# This prevents stale servers/tunnels from causing port and permission conflicts.

os.system("pkill -f 'mlflow server' || true")
os.system("pkill -f 'ngrok' || true")

print("Old MLflow/ngrok processes cleared.")

In [ ]:
# Start MLflow 3 tracking server.
#
# The CORS and allowed-host settings are required when the MLflow UI
# is accessed through an ngrok public URL from a browser.

log_file = open("/tmp/mlflow.log", "w")

mlflow_process = subprocess.Popen(
    [
        "mlflow",
        "server",
        "--backend-store-uri", DB_URI,
        "--default-artifact-root", str(ARTIFACT_DIR),
        "--serve-artifacts",
        "--host", MLFLOW_HOST,
        "--port", str(MLFLOW_PORT),
        "--allowed-hosts", "*",
        "--cors-allowed-origins", "*"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for the server to become available.
import urllib.request

server_ready = False
for _ in range(30):
    time.sleep(1)
    try:
        with urllib.request.urlopen(f"{MLFLOW_LOCAL_URL}/health", timeout=2) as response:
            if response.status == 200:
                server_ready = True
                break
    except Exception:
        pass

print("MLflow process running:", mlflow_process.poll() is None)
print("MLflow server ready:", server_ready)

if not server_ready:
    print("\nMLflow server log:")
    print(Path("/tmp/mlflow.log").read_text(errors="ignore"))
    raise RuntimeError(
        "MLflow server did not become ready. "
        "Check /tmp/mlflow.log before continuing."
    )

In [ ]:
print(Path("/tmp/mlflow.log").read_text(errors="ignore"))

In [ ]:
# Local health check
!curl -s http://127.0.0.1:5001/health

In [ ]:
# Create a fresh ngrok tunnel for the current Colab runtime.
#
# IMPORTANT:
# The previous ngrok error (ERR_NGROK_334) means the reserved/static endpoint
# was already online in another ngrok session. We therefore deliberately use
# a CLEAN ngrok config with NO endpoint/tunnel definition. This forces ngrok
# to create a new temporary public URL instead of reusing the old
# "erratic-graceless-regalia.ngrok-free.dev" endpoint.

from pyngrok import ngrok, conf
from getpass import getpass
import os
import time

# Stop only the local ngrok agent/process from this Colab runtime.
try:
    ngrok.kill()
except Exception:
    pass

os.system("pkill -f ngrok >/dev/null 2>&1 || true")
time.sleep(2)

# Use a completely separate config file so an old "pyngrok-default"
# endpoint/domain cannot be picked up from the persistent Colab config.
CLEAN_NGROK_CONFIG = "/tmp/mlflow_colab_ngrok.yml"
try:
    os.remove(CLEAN_NGROK_CONFIG)
except FileNotFoundError:
    pass

# Ask for the token only if it is not already available as an environment variable.
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTHTOKEN")

if not NGROK_AUTH_TOKEN:
    print("Enter your ngrok Auth Token securely (input will not be displayed):")
    NGROK_AUTH_TOKEN = getpass("ngrok Auth Token: ")

if not NGROK_AUTH_TOKEN:
    raise ValueError("An ngrok Auth Token is required.")

# config_version="2" + a clean config file prevents an existing
# pyngrok-default endpoint from being reused.
pyngrok_config = conf.PyngrokConfig(
    config_path=CLEAN_NGROK_CONFIG,
    auth_token=NGROK_AUTH_TOKEN,
    config_version="2",
    startup_timeout=30
)

# Create a NEW temporary public endpoint. Do NOT specify the old domain.
try:
    tunnel = ngrok.connect(
        addr=str(MLFLOW_PORT),
        proto="http",
        pyngrok_config=pyngrok_config
    )
except Exception as e:
    print("First ngrok connection attempt failed.")
    print("Checking for a stale local ngrok process and retrying once...")

    try:
        ngrok.kill()
    except Exception:
        pass

    os.system("pkill -f ngrok >/dev/null 2>&1 || true")
    time.sleep(3)

    tunnel = ngrok.connect(
        addr=str(MLFLOW_PORT),
        proto="http",
        pyngrok_config=pyngrok_config
    )

MLFLOW_PUBLIC_URL = tunnel.public_url

print("=" * 70)
print("MLflow Dashboard URL:")
print(MLFLOW_PUBLIC_URL)
print("=" * 70)
print(f"Forwarding to: http://127.0.0.1:{MLFLOW_PORT}")
print("Keep this Colab runtime running while using the dashboard.")


In [ ]:
# IMPORTANT:
# All Python MLflow operations use the tracking SERVER.
# Do not switch back to sqlite:///mlflow.db in later cells.

mlflow.set_tracking_uri(MLFLOW_LOCAL_URL)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

In [ ]:
# ngrok security note
#
# The notebook does not store the ngrok Auth Token in source code.
# A clean temporary config is used for this Colab runtime so that a
# previously configured/reserved endpoint cannot be reused accidentally.
#
# The dashboard URL is printed by the ngrok cell above.


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/data_employee_attrition.csv")

In [ ]:
df.head() # target feature is Attrition it has values Yes or No so it is classification problem

In [ ]:
df.tail()

In [ ]:
df.shape    # rows=1470, columns=35

In [ ]:
df.info()

In [ ]:
num_colmns = df.select_dtypes(include=['number'])
cat_colmns = df.select_dtypes(include=['object'])

In [ ]:
df.describe()

'''
By observing the mean and 50% the feature 'DistanceFromHome' is right skwed and 'EmployeeNumber' is also little right skwed.
The 'TotalWorkingYears', 'YearsAtCompany', 'YearsSinceLastPromotion' and 'YearsWithCurrManager' its 75% amd max values comparionsion is very high, it may containe outliers.

'''

In [ ]:
df.describe(include="object")

#The featute Over18 has only 1 unique value so it will not contribute any thing for prediction, we can drop this feature

In [ ]:
df.columns

In [ ]:
df.isnull().sum()   #There is no missing values present in the dateset

In [ ]:
duplicates = df.duplicated().sum()  #There are no duplicates present in the dataset
print(duplicates)

In [ ]:
df.nunique()

# 'StandardHours ,'Over18' and  'EmployeeCount' has only 1 unique vale it will not contribute for prediction so we can drop these 3 features

In [ ]:
attrition_percent = (
    df["Attrition"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(attrition_percent)

plt.figure(figsize=(6, 4))
sns.barplot(x=attrition_percent.index, y=attrition_percent.values)
plt.title("Attrition Class Distribution")
plt.xlabel("Attrition")
plt.ylabel("Percentage")
plt.show()

# #Target feature has an imbalanced data

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["Age"], kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
df["Department"].value_counts().plot(kind="bar")

In [ ]:
df.corr(numeric_only=True)   # bivariate analysis Numerical vs Numerical

In [ ]:
plt.figure(figsize=(18,12))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", linewidths=0.5, annot_kws={"size":8})
plt.show()

In [ ]:
# highly correlated features
corr = df.corr(numeric_only=True)

plt.figure(figsize=(18, 12))
sns.heatmap(
    corr[(corr.abs() > 0.5)],
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.tight_layout()
plt.show()

In [ ]:
num_colmns = df.select_dtypes(include='number').columns
plt.figure(figsize=(20, 10))
sns.boxplot(data=df[num_colmns])

plt.xticks(rotation=90)
plt.title("Boxplot of All Numerical Features")
plt.show()

As per above boxplot observation MonthlyIncome feature has outliers

In [ ]:
outlier_summary = []

for col in num_colmns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary.append({
        "Column": col,
        "Outlier Count": count,
        "Outlier %": round((count / len(df)) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df.sort_values("Outlier Count", ascending=False)

In [ ]:
df.skew(numeric_only=True) # To check skew in the data

In [ ]:
skewness = df.skew(numeric_only=True)

for col in skewness[abs(skewness) > 0.5].index:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"{col} (Skewness = {skewness[col]:.2f})")
    plt.show()

In [ ]:
df['Attrition'] = df['Attrition'].map({'No': 0, 'Yes': 1}) # converting Attrition feature into numarical

##Observation
1.Dataset contains 1470 rows and 35 columns.

2.Outliers are present in these features TrainingTimesLastYear,PerformanceRating, MonthlyIncome, YearsSinceLastPromotion, YearsAtCompany, StockOptionLevel, TotalWorkingYears,NumCompaniesWorked, YearsInCurrentRole and YearsWithCurrManager

3.Target feature has an imbalanced data (83.87% and 16.13%)

4.'StandardHours ,'Over18' and  'EmployeeCount' has only 1 unique value it will not contribute for prediction so we can drop these 3 features

5.By observing the mean and 50% the feature 'DistanceFromHome' is right skwed and 'EmployeeNumber' is also little right skwed.
The 'TotalWorkingYears', 'YearsAtCompany', 'YearsSinceLastPromotion' and 'YearsWithCurrManager' its 75% amd max values comparionsion is very high, it may containe outliers.

6.DistanceFromHome,MonthlyIncome, NumCompaniesWorked, PerformanceRating, TotalWorkingYears, YearsAtCompany, YearsSinceLastPromotion and YearsWithCurrManager are highly right skewd



###Feature Engineering

In [ ]:
df.drop(columns=['EmployeeNumber', 'Over18', 'StandardHours'], inplace=True)

In [ ]:
df.shape

In [ ]:
# Columns to check for outliers
columns = ["TrainingTimesLastYear","PerformanceRating", "MonthlyIncome", "YearsSinceLastPromotion", "YearsAtCompany", "StockOptionLevel", "TotalWorkingYears","NumCompaniesWorked", "YearsInCurrentRole", "YearsWithCurrManager"]

# Create a copy of the dataframe
df_clean = df.copy()

for col in columns:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Remove outliers
    df_clean = df_clean[
        (df_clean[col] >= lower_bound) &
        (df_clean[col] <= upper_bound)
    ]

print("Original Shape:", df.shape)
print("Shape After Removing Outliers:", df_clean.shape)

In [ ]:
# Handling the skewed numerical features using Yeo-Johnson transformation.
from sklearn.preprocessing import PowerTransformer

cols = [
    "DistanceFromHome",
    "MonthlyIncome",
    "JobLevel",
    "PerformanceRating",
    "NumCompaniesWorked",
    "TotalWorkingYears",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]

pt = PowerTransformer(method="yeo-johnson")
df[cols] = pt.fit_transform(df[cols])

print("Yeo-Johnson transformation applied to:")
print(cols)

In [ ]:
df.select_dtypes(include='object').columns

In [ ]:
df['OverTime'].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cat_cols = df.select_dtypes(include='object').columns

#for col in cat_cols:
    #df[col] = le.fit_transform(df[col])

In [ ]:
#converting object features into numarical features
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Gender'] = le.fit_transform(df['Gender'])
df['OverTime'] = le.fit_transform(df['OverTime'])

In [ ]:
# Converting categorical features into numerical features.
#
# The original notebook attempted to assign the full OneHotEncoder matrix
# back into one dataframe column. That is not valid when a column has more
# than two categories. The corrected implementation preserves the intended
# one-hot encoding with drop_first=True.

from sklearn.preprocessing import OneHotEncoder

cat_cols = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "JobRole",
    "MaritalStatus"
]

encoder = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_array = encoder.fit_transform(df[cat_cols])

encoded_columns = encoder.get_feature_names_out(cat_cols)

encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoded_columns,
    index=df.index
)

df = pd.concat(
    [
        df.drop(columns=cat_cols),
        encoded_df
    ],
    axis=1
)

print("One-hot encoded columns:")
print(encoded_columns.tolist())
print("New dataframe shape:", df.shape)

In [ ]:
df.drop(columns=['EmployeeCount'], inplace=True)

In [ ]:
df.shape

In [ ]:
# Feature selection using Mutual Information.
from sklearn.feature_selection import mutual_info_classif

X = df.drop("Attrition", axis=1)
y = df["Attrition"]

mi = mutual_info_classif(
    X,
    y,
    random_state=50
)

mi_scores = (
    pd.Series(mi, index=X.columns)
    .sort_values(ascending=False)
)

print(mi_scores)

In [ ]:
import matplotlib.pyplot as plt

mi_scores.plot(kind="bar", figsize=(12,6))
plt.title("Mutual Information Scores")
plt.ylabel("MI Score")
plt.show()

In [ ]:
selected_features = mi_scores[mi_scores > 0].index

X_selected = X[selected_features]

print(selected_features)

In [ ]:
print(X.shape)
print(X_selected.shape)

#Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Final independent test set.
# Validation will be performed later using stratified 5-fold CV on X_train.
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=50,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("Testing target distribution:")
print(y_test.value_counts(normalize=True).round(3))

#Train a Model

In [ ]:
# Validation strategy:
# - Training: X_train / y_train
# - Validation: 5-fold Stratified Cross-Validation on X_train / y_train
# - Final testing: untouched X_test / y_test
#
# We use explicit MLflow logging instead of mlflow.autolog() so that every
# parameter, validation metric, test metric, visualization, and model is
# easy to identify in the MLflow UI.

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)
from mlflow.models import infer_signature

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

PLOTS_DIR = Path("/content/mlflow_plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("5-fold stratified cross-validation configured.")

In [ ]:
# ============================================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=10000)

with mlflow.start_run(run_name="LogisticRegression") as run:

    # -------------------------
    # Validation
    # -------------------------
    cv_results = cross_validate(
        log_reg,
        X_train,
        y_train,
        cv=CV,
        scoring=["accuracy", "roc_auc", "precision", "recall", "f1"],
        n_jobs=-1
    )

    val_accuracy_mean = cv_results["test_accuracy"].mean()
    val_accuracy_std = cv_results["test_accuracy"].std()
    val_auc_mean = cv_results["test_roc_auc"].mean()
    val_auc_std = cv_results["test_roc_auc"].std()

    # -------------------------
    # Train final model
    # -------------------------
    log_reg.fit(X_train, y_train)

    y_pred = log_reg.predict(X_test)
    y_prob = log_reg.predict_proba(X_test)[:, 1]

    test_accuracy = accuracy_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred, zero_division=0)
    test_recall = recall_score(y_test, y_pred, zero_division=0)
    test_f1 = f1_score(y_test, y_pred, zero_division=0)
    test_auc = roc_auc_score(y_test, y_prob)

    print("Logistic Regression")
    print("Validation Accuracy:", f"{val_accuracy_mean:.4f} ± {val_accuracy_std:.4f}")
    print("Validation AUC     :", f"{val_auc_mean:.4f} ± {val_auc_std:.4f}")
    print("Test Accuracy      :", f"{test_accuracy:.4f}")
    print("Test AUC           :", f"{test_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # -------------------------
    # MLflow parameters/metrics
    # -------------------------
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 10000)
    mlflow.log_param("validation_folds", 5)

    mlflow.log_metric("validation_accuracy_mean", val_accuracy_mean)
    mlflow.log_metric("validation_accuracy_std", val_accuracy_std)
    mlflow.log_metric("validation_auc_mean", val_auc_mean)
    mlflow.log_metric("validation_auc_std", val_auc_std)

    mlflow.log_metric("accuracy", test_accuracy)
    mlflow.log_metric("precision", test_precision)
    mlflow.log_metric("recall", test_recall)
    mlflow.log_metric("f1_score", test_f1)
    mlflow.log_metric("auc", test_auc)

    # -------------------------
    # ROC visualization
    # -------------------------
    log_fpr, log_tpr, log_thresholds = roc_curve(y_test, y_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(
        log_fpr,
        log_tpr,
        label=f"AUC = {test_auc:.3f}"
    )
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Logistic Regression ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()

    log_roc_path = PLOTS_DIR / "logistic_regression_roc.png"
    plt.savefig(log_roc_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(log_roc_path), artifact_path="visualizations")

    # -------------------------
    # Confusion matrix
    # -------------------------
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Logistic Regression Confusion Matrix")
    plt.tight_layout()

    log_cm_path = PLOTS_DIR / "logistic_regression_confusion_matrix.png"
    plt.savefig(log_cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(log_cm_path), artifact_path="visualizations")

    # -------------------------
    # Model signature + input example
    # -------------------------
    signature = infer_signature(
        X_train.head(5),
        log_reg.predict(X_train.head(5))
    )

    model_input_example = X_train.head(5)

    model_info_log = mlflow.sklearn.log_model(
        log_reg,
        name="logistic_regression_model",
        signature=signature,
        input_example=model_input_example
    )

    print("Run ID:", run.info.run_id)
    print("Logged Model ID:", model_info_log.model_id)

In [ ]:
# ============================================================
# MODEL 2: RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

model2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=50
)

with mlflow.start_run(run_name="Random_Forest_Mdl") as run:

    # -------------------------
    # Validation
    # -------------------------
    cv_results = cross_validate(
        model2,
        X_train,
        y_train,
        cv=CV,
        scoring=["accuracy", "roc_auc", "precision", "recall", "f1"],
        n_jobs=-1
    )

    rf_val_accuracy_mean = cv_results["test_accuracy"].mean()
    rf_val_accuracy_std = cv_results["test_accuracy"].std()
    rf_val_auc_mean = cv_results["test_roc_auc"].mean()
    rf_val_auc_std = cv_results["test_roc_auc"].std()

    # -------------------------
    # Train final model
    # -------------------------
    model2.fit(X_train, y_train)

    y_pred2 = model2.predict(X_test)
    y_prob2 = model2.predict_proba(X_test)[:, 1]

    rf_accuracy = accuracy_score(y_test, y_pred2)
    rf_precision = precision_score(y_test, y_pred2, zero_division=0)
    rf_recall = recall_score(y_test, y_pred2, zero_division=0)
    rf_f1 = f1_score(y_test, y_pred2, zero_division=0)
    rf_auc = roc_auc_score(y_test, y_prob2)

    print("Random Forest")
    print("Validation Accuracy:", f"{rf_val_accuracy_mean:.4f} ± {rf_val_accuracy_std:.4f}")
    print("Validation AUC     :", f"{rf_val_auc_mean:.4f} ± {rf_val_auc_std:.4f}")
    print("Test Accuracy      :", f"{rf_accuracy:.4f}")
    print("Test AUC           :", f"{rf_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred2, zero_division=0))

    # -------------------------
    # MLflow parameters/metrics
    # -------------------------
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("random_state", 50)
    mlflow.log_param("validation_folds", 5)

    mlflow.log_metric("validation_accuracy_mean", rf_val_accuracy_mean)
    mlflow.log_metric("validation_accuracy_std", rf_val_accuracy_std)
    mlflow.log_metric("validation_auc_mean", rf_val_auc_mean)
    mlflow.log_metric("validation_auc_std", rf_val_auc_std)

    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("f1_score", rf_f1)
    mlflow.log_metric("auc", rf_auc)

    # -------------------------
    # ROC
    # -------------------------
    rf_fpr, rf_tpr, rf_thresholds = roc_curve(y_test, y_prob2)

    # -------------------------
    # Confusion matrix
    # -------------------------
    rf_cm = confusion_matrix(y_test, y_pred2)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        rf_cm,
        annot=True,
        fmt="d",
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Random Forest Confusion Matrix")
    plt.tight_layout()

    rf_cm_path = PLOTS_DIR / "random_forest_confusion_matrix.png"
    plt.savefig(rf_cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(rf_cm_path), artifact_path="visualizations")

    # -------------------------
    # Feature importance
    # -------------------------
    rf_importance = (
        pd.Series(model2.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .head(15)
    )

    plt.figure(figsize=(10, 6))
    rf_importance.sort_values().plot(kind="barh")
    plt.title("Random Forest - Top 15 Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()

    rf_fi_path = PLOTS_DIR / "random_forest_feature_importance.png"
    plt.savefig(rf_fi_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(rf_fi_path), artifact_path="visualizations")

    # -------------------------
    # Model signature + input example
    # -------------------------
    signature = infer_signature(
        X_train.head(5),
        model2.predict(X_train.head(5))
    )

    model_info_rf = mlflow.sklearn.log_model(
        model2,
        name="RandomForest_model",
        signature=signature,
        input_example=X_train.head(5)
    )

    print("Run ID:", run.info.run_id)
    print("Logged Model ID:", model_info_rf.model_id)

In [ ]:
# Random Forest ROC Curve

plt.figure(figsize=(8, 6))

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"AUC = {rf_auc:.3f}"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest ROC Curve")

plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MODEL 3: XGBOOST
# ============================================================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=50,
    eval_metric="auc"
)

with mlflow.start_run(run_name="XGBoost_Advanced") as run:

    # -------------------------
    # Validation
    # -------------------------
    cv_results = cross_validate(
        xgb_model,
        X_train,
        y_train,
        cv=CV,
        scoring=["accuracy", "roc_auc", "precision", "recall", "f1"],
        n_jobs=-1
    )

    xgb_val_accuracy_mean = cv_results["test_accuracy"].mean()
    xgb_val_accuracy_std = cv_results["test_accuracy"].std()
    xgb_val_auc_mean = cv_results["test_roc_auc"].mean()
    xgb_val_auc_std = cv_results["test_roc_auc"].std()

    # -------------------------
    # Train final model
    # -------------------------
    xgb_model.fit(X_train, y_train)

    y_pred3 = xgb_model.predict(X_test)
    y_prob3 = xgb_model.predict_proba(X_test)[:, 1]

    xgb_accuracy = accuracy_score(y_test, y_pred3)
    xgb_precision = precision_score(y_test, y_pred3, zero_division=0)
    xgb_recall = recall_score(y_test, y_pred3, zero_division=0)
    xgb_f1 = f1_score(y_test, y_pred3, zero_division=0)
    xgb_auc = roc_auc_score(y_test, y_prob3)

    print("XGBoost")
    print("Validation Accuracy:", f"{xgb_val_accuracy_mean:.4f} ± {xgb_val_accuracy_std:.4f}")
    print("Validation AUC     :", f"{xgb_val_auc_mean:.4f} ± {xgb_val_auc_std:.4f}")
    print("Test Accuracy      :", f"{xgb_accuracy:.4f}")
    print("Test AUC           :", f"{xgb_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred3, zero_division=0))

    # -------------------------
    # MLflow parameters/metrics
    # -------------------------
    mlflow.log_param("model_type", "XGBClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("random_state", 50)
    mlflow.log_param("eval_metric", "auc")
    mlflow.log_param("validation_folds", 5)

    mlflow.log_metric("validation_accuracy_mean", xgb_val_accuracy_mean)
    mlflow.log_metric("validation_accuracy_std", xgb_val_accuracy_std)
    mlflow.log_metric("validation_auc_mean", xgb_val_auc_mean)
    mlflow.log_metric("validation_auc_std", xgb_val_auc_std)

    mlflow.log_metric("accuracy", xgb_accuracy)
    mlflow.log_metric("precision", xgb_precision)
    mlflow.log_metric("recall", xgb_recall)
    mlflow.log_metric("f1_score", xgb_f1)
    mlflow.log_metric("auc", xgb_auc)

    # -------------------------
    # ROC
    # -------------------------
    xgb_fpr, xgb_tpr, xgb_thresholds = roc_curve(y_test, y_prob3)

    # -------------------------
    # Confusion matrix
    # -------------------------
    xgb_cm = confusion_matrix(y_test, y_pred3)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        xgb_cm,
        annot=True,
        fmt="d",
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("XGBoost Confusion Matrix")
    plt.tight_layout()

    xgb_cm_path = PLOTS_DIR / "xgboost_confusion_matrix.png"
    plt.savefig(xgb_cm_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(xgb_cm_path), artifact_path="visualizations")

    # -------------------------
    # Feature importance
    # -------------------------
    xgb_importance = (
        pd.Series(xgb_model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .head(15)
    )

    plt.figure(figsize=(10, 6))
    xgb_importance.sort_values().plot(kind="barh")
    plt.title("XGBoost - Top 15 Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()

    xgb_fi_path = PLOTS_DIR / "xgboost_feature_importance.png"
    plt.savefig(xgb_fi_path, dpi=150, bbox_inches="tight")
    plt.show()
    mlflow.log_artifact(str(xgb_fi_path), artifact_path="visualizations")

    # -------------------------
    # Model signature + input example
    # -------------------------
    signature = infer_signature(
        X_train.head(5),
        xgb_model.predict(X_train.head(5))
    )

    model_info_xgb = mlflow.sklearn.log_model(
        xgb_model,
        name="XGBoost_model",
        signature=signature,
        input_example=X_train.head(5)
    )

    print("Run ID:", run.info.run_id)
    print("Logged Model ID:", model_info_xgb.model_id)

In [ ]:
# XGBoost ROC Curve

plt.figure(figsize=(8, 6))

plt.plot(
    xgb_fpr,
    xgb_tpr,
    label=f"AUC = {xgb_auc:.3f}"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost ROC Curve")

plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Model Comparison, Validation and Final Model Selection

The three models are compared using:
- 5-fold stratified cross-validation on the training data (validation)
- Independent test-set accuracy/AUC/precision/recall/F1
- ROC curves and confusion matrices

The model with the highest **test accuracy** is selected for Model Registry. The validation metrics are retained in MLflow for reproducibility.

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Validation Accuracy Mean": [
        val_accuracy_mean,
        rf_val_accuracy_mean,
        xgb_val_accuracy_mean
    ],
    "Validation Accuracy Std": [
        val_accuracy_std,
        rf_val_accuracy_std,
        xgb_val_accuracy_std
    ],
    "Validation AUC Mean": [
        val_auc_mean,
        rf_val_auc_mean,
        xgb_val_auc_mean
    ],
    "Test Accuracy": [
        test_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "Test Precision": [
        test_precision,
        rf_precision,
        xgb_precision
    ],
    "Test Recall": [
        test_recall,
        rf_recall,
        xgb_recall
    ],
    "Test F1": [
        test_f1,
        rf_f1,
        xgb_f1
    ],
    "Test AUC": [
        test_auc,
        rf_auc,
        xgb_auc
    ]
})

print(results.round(4))

# -------------------------
# Accuracy comparison
# -------------------------
plt.figure(figsize=(9, 5))
sns.barplot(
    data=results,
    x="Model",
    y="Test Accuracy"
)
plt.ylim(0, 1)
plt.title("Test Accuracy Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# -------------------------
# AUC comparison
# -------------------------
plt.figure(figsize=(9, 5))
sns.barplot(
    data=results,
    x="Model",
    y="Test AUC"
)
plt.ylim(0, 1)
plt.title("Test AUC Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# -------------------------
# Combined ROC comparison
# -------------------------
plt.figure(figsize=(9, 7))

plt.plot(
    log_fpr,
    log_tpr,
    label=f"Logistic Regression (AUC={test_auc:.3f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC={rf_auc:.3f})"
)

plt.plot(
    xgb_fpr,
    xgb_tpr,
    label=f"XGBoost (AUC={xgb_auc:.3f})"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# -------------------------
# Save comparison results
# -------------------------
comparison_path = PLOTS_DIR / "model_comparison.csv"
results.to_csv(comparison_path, index=False)

print("\nBest model by test accuracy:")
best_model_name = results.loc[
    results["Test Accuracy"].idxmax(),
    "Model"
]
print(best_model_name)

In [ ]:
# Log the overall comparison as a dedicated MLflow run.
with mlflow.start_run(run_name="Model_Comparison") as comparison_run:

    for _, row in results.iterrows():
        safe_name = (
            row["Model"]
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        mlflow.log_metric(
            f"{safe_name}_test_accuracy",
            float(row["Test Accuracy"])
        )
        mlflow.log_metric(
            f"{safe_name}_test_auc",
            float(row["Test AUC"])
        )
        mlflow.log_metric(
            f"{safe_name}_validation_accuracy_mean",
            float(row["Validation Accuracy Mean"])
        )

    mlflow.log_artifact(
        str(comparison_path),
        artifact_path="model_comparison"
    )

    mlflow.set_tag("selection_metric", "test_accuracy")
    mlflow.set_tag("best_model", best_model_name)

print("Model comparison logged to MLflow.")

# MLflow

In [ ]:
# Process cleanup is handled automatically by the MLflow setup cell above.
# This cell is intentionally kept from the original notebook for continuity.
print("MLflow process cleanup is already handled.")

In [ ]:
# The original experimental MLflow snippet is replaced by the explicit
# three-model logging and registry workflow above.
print("Legacy MLflow example retained as a note; no action required.")

In [ ]:
# ============================================================
# MLFLOW MODEL REGISTRY - MLflow 3 SAFE WORKFLOW
# ============================================================

import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

# Always communicate through the MLflow tracking server.
mlflow.set_tracking_uri(MLFLOW_LOCAL_URL)

client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())

# ------------------------------------------------------------
# 1. Get experiment
# ------------------------------------------------------------
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(
        f"Experiment '{EXPERIMENT_NAME}' was not found."
    )

experiment_id = experiment.experiment_id

print("Experiment ID:", experiment_id)

# ------------------------------------------------------------
# 2. Find the best model using test accuracy
# ------------------------------------------------------------
runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id]
)

accuracy_col = "metrics.accuracy"

if accuracy_col not in runs_df.columns:
    raise ValueError(
        f"'{accuracy_col}' was not found in MLflow runs."
    )

runs_df[accuracy_col] = pd.to_numeric(
    runs_df[accuracy_col],
    errors="coerce"
)

candidate_runs = runs_df.dropna(
    subset=[accuracy_col]
).copy()

# Exclude the comparison run because it contains comparison metrics,
# not a single model accuracy metric.
candidate_runs = candidate_runs[
    candidate_runs["tags.mlflow.runName"] != "Model_Comparison"
].copy()

if candidate_runs.empty:
    raise ValueError("No valid model runs with accuracy were found.")

candidate_runs = candidate_runs.sort_values(
    by=accuracy_col,
    ascending=False
)

best_run = candidate_runs.iloc[0]

best_run_id = best_run["run_id"]
best_run_name = best_run["tags.mlflow.runName"]
best_score = float(best_run[accuracy_col])

print("\n" + "=" * 70)
print("BEST MODEL RUN")
print("=" * 70)
print("Run Name :", best_run_name)
print("Run ID   :", best_run_id)
print("Accuracy :", f"{best_score:.4f}")

# ------------------------------------------------------------
# 3. Find the MLflow 3 Logged Model for this run
# ------------------------------------------------------------
logged_models = mlflow.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"source_run_id = '{best_run_id}'",
    output_format="list"
)

if not logged_models:
    raise ValueError(
        "No Logged Model was found for the winning run. "
        "Make sure mlflow.sklearn.log_model() completed successfully."
    )

# Prefer READY models.
ready_models = [
    m for m in logged_models
    if str(m.status).upper().endswith("READY")
]

selected_logged_model = (
    ready_models[0]
    if ready_models
    else logged_models[0]
)

model_id = selected_logged_model.model_id
logged_model_name = selected_logged_model.name

print("\n" + "=" * 70)
print("LOGGED MODEL")
print("=" * 70)
print("Logged Model Name:", logged_model_name)
print("Model ID         :", model_id)
print("Status            :", selected_logged_model.status)

# ------------------------------------------------------------
# 4. MLflow 3 model-id URI
# ------------------------------------------------------------
model_uri = f"models:/{model_id}"

print("Model URI:", model_uri)

# ------------------------------------------------------------
# 5. Register model
# ------------------------------------------------------------
registered_model_name = "Best_Attrition_Prediction_Model"

model_details = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("\n" + "=" * 70)
print("MODEL REGISTRATION SUCCESSFUL")
print("=" * 70)
print("Registered Model:", model_details.name)
print("Version         :", model_details.version)
print("Best Accuracy   :", f"{best_score:.4f}")

# ------------------------------------------------------------
# 6. Add useful version tags
# ------------------------------------------------------------
client.set_model_version_tag(
    name=registered_model_name,
    version=model_details.version,
    key="model_type",
    value=best_model_name
)

client.set_model_version_tag(
    name=registered_model_name,
    version=model_details.version,
    key="test_accuracy",
    value=f"{best_score:.6f}"
)

client.set_model_version_tag(
    name=registered_model_name,
    version=model_details.version,
    key="source_run_id",
    value=best_run_id
)

# ------------------------------------------------------------
# 7. Set 'champion' alias to the newly registered best model
# ------------------------------------------------------------
try:
    client.set_registered_model_alias(
        registered_model_name,
        "champion",
        model_details.version
    )
    print(
        f"Alias 'champion' -> version {model_details.version}"
    )
except Exception as alias_error:
    print("Alias could not be set:", alias_error)

# ------------------------------------------------------------
# 8. Verify the registry
# ------------------------------------------------------------
registered = client.get_registered_model(
    registered_model_name
)

print("\nRegistered model:", registered.name)

for version in client.search_model_versions(
    f"name = '{registered_model_name}'"
):
    print(
        f"Version {version.version} | "
        f"Status={version.status} | "
        f"Run={version.run_id}"
    )

In [ ]:
# ============================================================
# FINAL MODEL REGISTRY VERIFICATION / LOAD TEST
# ============================================================

mlflow.set_tracking_uri(MLFLOW_LOCAL_URL)

client = MlflowClient()

registered_model_name = "Best_Attrition_Prediction_Model"

# Get champion alias if available.
try:
    champion = client.get_model_version_by_alias(
        registered_model_name,
        "champion"
    )

    champion_uri = f"models:/{registered_model_name}@champion"

    print("Champion model version:", champion.version)
    print("Champion model URI:", champion_uri)

    loaded_model = mlflow.sklearn.load_model(champion_uri)

    sample_predictions = loaded_model.predict(
        X_test.head(10)
    )

    print("Sample predictions:", sample_predictions)

except Exception as e:
    print("Champion alias verification was not available:", e)

# Ensure no active run remains open.
if mlflow.active_run() is not None:
    mlflow.end_run()

print("\nMLflow workflow completed successfully.")
print("Dashboard URL:", MLFLOW_PUBLIC_URL)

## MLflow Dashboard

After running the MLflow setup cells, open the printed:

**MLflow Dashboard URL**

It will look like:

`https://<random-name>.ngrok-free.dev`

Do not use `http://127.0.0.1:5001` from your desktop browser; that address exists inside the Colab runtime.

### MLflow objects created by this notebook

- Experiment: `employee_attrition`
- Runs: Logistic Regression, Random Forest, XGBoost, Model Comparison
- Logged Models: one for each trained model
- Metrics: validation accuracy/AUC plus test accuracy/precision/recall/F1/AUC
- Artifacts: ROC curves, confusion matrices, feature-importance plots, model comparison CSV
- Registered Model: `Best_Attrition_Prediction_Model`
- Alias: `champion` for the selected best model version